# All Fastchem Gas and Cond

Hajime Kawahara 2025/11/27



In [1]:
from jax import config
config.update("jax_enable_x64", True)

We assume N2+H2O (gas, water, ice) system using fastchem/fastchem_cond presets. 

In [2]:
from exogibbs.presets.fastchem_cond import chemsetup as condsetup
cond = condsetup()
from exogibbs.presets.fastchem import chemsetup as gassetup
gas = gassetup()


fastchem_cond presets in ExoGibbs
number of species: 186 elements: 28 molecules: 186
fastchem presets in ExoGibbs
number of species: 523 elements: 28 molecules: 495


In [3]:
import numpy as np
from exojax.utils.zsol import nsol
import jax.numpy as jnp

solar_abundance = nsol()
nsol_vector = jnp.array(
    [solar_abundance[el] for el in gas.elements[:-1]]
)  # no solar abundance for e-
element_vector = jnp.append(nsol_vector, 0.0)

formula_matrix_gas = gas.formula_matrix

print("Formula matrix (gas):")
print(formula_matrix_gas)

formula_matrix_cond = cond.formula_matrix

print("Formula matrix (cond):")
print(formula_matrix_cond)

b_ref = gas.element_vector_reference

Database for solar abundance =  AAG21
Asplund, M., Amarsi, A. M., & Grevesse, N. 2021, arXiv:2105.01661
Formula matrix (gas):
[[ 1  0  0 ...  0  0  0]
 [ 0  1  0 ...  0  0  0]
 [ 0  0  1 ...  0  0  0]
 ...
 [ 0  0  0 ...  1  0  0]
 [ 0  0  0 ...  0  1  1]
 [ 0  0  0 ...  1 -1  1]]
Formula matrix (cond):
[[1 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


This setting yields rank(Ac, Ag) < |b_element| because formula_matrix_gas[:,0] = formula_matrix_cond[:,0]. We need to redefine the formulation using the matrix contraction. 

In [4]:
from exogibbs.thermo.stoichiometry import contract_formula_matrix
formula_matrix_gas_eff, formula_matrix_cond_eff, indep_element_mask = contract_formula_matrix(formula_matrix_gas, formula_matrix_cond)
#elements_eff =elements[indep_element_mask]

print("Formula matrix (gas):")
print(formula_matrix_gas_eff)
print("Formula matrix (cond):")
print(formula_matrix_cond_eff)
#print("independent elements:")
#print(elements_eff)


No contraction of the system needed.
Formula matrix (gas):
[[ 1  0  0 ...  0  0  0]
 [ 0  1  0 ...  0  0  0]
 [ 0  0  1 ...  0  0  0]
 ...
 [ 0  0  0 ...  1  0  0]
 [ 0  0  0 ...  0  1  1]
 [ 0  0  0 ...  1 -1  1]]
Formula matrix (cond):
[[1 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


Output the reference-state value of ( $h = \mu / (RT)$ ) at temperature ( T ).


## minimization using minimize_gibbs_cond_core

In [ ]:
from exogibbs.optimize.pipm_rgie_cond import minimize_gibbs_cond_core
import jax.numpy as jnp
from exogibbs.api.chemistry import ThermoState

from exogibbs.optimize.core import compute_ln_normalized_pressure


In [6]:

# Thermodynamic conditions
P = 1.0  # bar
Pref = 1.0  # bar, reference pressure
ln_normalized_pressure = compute_ln_normalized_pressure(P, Pref)

# Initial guess for log number densities
ln_nk = jnp.zeros(formula_matrix_gas_eff.shape[1])  # log(n_species)
ln_mk = jnp.zeros(formula_matrix_cond_eff.shape[1])   # log(n_condensates)
ln_ntot = jnp.log(jnp.sum(jnp.exp(ln_nk)))  # log(total number density)

epsilon = 1.0
epsilon_crit = 0.8

for i, temperature in enumerate([200.0]):

    #PD-IPM
    nkpath=[]
    mkpath=[]
    eppath=[]
    
    thermo_state = ThermoState(
        temperature=temperature,
        ln_normalized_pressure=ln_normalized_pressure,
        element_vector=b_ref,
    )



    while epsilon > epsilon_crit:
        epsilon = epsilon - 0.1

        ln_nk, ln_mk, ln_ntot, counter = minimize_gibbs_cond_core(
            thermo_state,
            ln_nk_init=ln_nk,
            ln_mk_init=ln_mk,
            ln_ntot_init=ln_ntot,
            formula_matrix=formula_matrix_gas_eff,
            formula_matrix_cond=formula_matrix_cond_eff,
            hvector_func=gas.hvector_func,
            hvector_cond_func=cond.hvector_func,
            epsilon=epsilon,  ### new argument
            residual_crit=1.0e-10,
            max_iter=100,
        )

        nkpath.append(jnp.exp(ln_nk)[0])
        mkpath.append(jnp.exp(ln_mk)[0])
        eppath.append(epsilon)
        print("Optimization:", ln_nk, "counter=", counter, "epsilon=", epsilon)
    

    

Optimization: [-2.75830302e+01  8.78824328e-01 -1.97064047e+01 -2.38175038e+01
 -7.65661126e+00 -1.72740070e+01 -1.89393759e+01 -1.15934661e+01
 -1.23690969e+01 -1.78639506e+01 -4.63931115e-02 -1.01922032e+01
  1.17485094e+00 -1.51435984e+01 -2.10968348e+01 -1.53903107e+01
 -1.46494619e+01 -1.59134476e+01  1.05473148e+00 -1.74899470e+01
 -1.53941295e+01 -1.64269864e+01 -9.39087951e+00 -2.55116606e+01
 -2.96338745e+01 -2.33941467e+01 -8.17950738e+00 -3.19426893e+01
 -1.85757074e+01 -1.38788029e+01 -7.44637857e+00 -1.63255401e+01
 -1.37310324e+01 -7.23743236e+00 -7.02618334e+00 -1.86210555e+01
 -1.58392884e+01 -1.42071708e+01 -8.70992310e+00 -7.64721523e+00
 -7.90155722e+00 -2.68960530e+01 -2.65370156e+01 -2.07181233e+01
 -1.88217975e+01 -3.15829259e+01 -2.60882271e+01 -2.76784943e+01
 -2.39740517e+01 -4.74649202e+01 -7.91354403e+00 -6.80833316e+00
 -3.60795733e+01 -3.50789077e+01 -3.45808371e+01 -1.52255133e+01
 -3.11241254e+00 -4.53656509e+00 -4.30636254e+00 -5.55493701e+00
 -9.5474743